# stress_hard rerun (2026-09-15)

Fresh T4 runtime required (no `/content/berlin-marso-hackathon`, no `/content/marso-py312`).
Run cells **one at a time, top to bottom**. Do not Run All.

1. Prerequisites: mount Drive, restore the 46 reviewed sources (zip `3a4c5e71…`), build Python 3.12 + CUDA env, restore the three best checkpoints by SHA.
2. Launch: dry-run the durable wrapper with `--stages stress/hard`, then start it inside tmux (results sync to `MyDrive/marso/validations/durable_*`).
3. Status: read local status/child log; rerun any time.


In [2]:
import os, sys, shutil, subprocess, hashlib
from pathlib import Path
print('TMUX', shutil.which('tmux'), flush=True)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout, flush=True)
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', timeout_ms=600000)
assert os.path.ismount('/content/drive')
print('DRIVE_MOUNTED', flush=True)
for filename, expected in [('recovery_environment_cell_20260915b.py','633d0a4357e73c3e30d4afe530b980ef3169df277483d0274e6e599938714061'),
                           ('recovery_inputs_cell_20260915.py','dfd1bd437a849fc2547b41be75e6905cee6fa711392e0838904c377feda78a12')]:
    source_file = Path('/content/drive/MyDrive/marso') / filename
    payload = source_file.read_bytes()
    assert hashlib.sha256(payload).hexdigest() == expected, filename
    exec(compile(payload, str(source_file), 'exec'))
print('STRESS_HARD_PREREQUISITES_READY', flush=True)


Mounted at /content/drive
DRIVE_MOUNTED
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 91.3 MB/s eta 0:00:00
Installed Python 3.12.3 in 2.05s
 + cpython-3.12.3-linux-x86_64-gnu (python3.12)
Using CPython 3.12.3
Creating virtual environment with seed packages at: marso-py312
 + pip==26.2.1
Activate with: source marso-py312/bin/activate
Using Python 3.12.3 environment at: marso-py312
Resolved 32 packages in 946ms
Prepared 32 packages in 1m 05s
Installed 32 packages in 310ms
 + cuda-bindings==12.9.7
 + cuda-pathfinder==1.6.0
 + cuda-toolkit==12.8.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==1.13.1.3
 + nvidia-curand-cu12==10.3.9.90
 + nvidia-cusolver-cu12==11.7.3.90
 + nv

In [3]:
from pathlib import Path
import hashlib
p = Path('/content/drive/MyDrive/marso/tmux_launch_cell_stress_hard_20260915.py')
s = p.read_bytes()
assert hashlib.sha256(s).hexdigest() == 'bd5c0dc8b86324314851b9c273815c206cbeb2bd9a24bffaa5973d9cef78a1e6'
exec(compile(s, str(p), 'exec'))


DURABLE_DRY_RUN_PASSED {"status": "dry_run", "stages": ["stress/hard"], "episodes": 100, "wrapper_sha256": "0f1ee62503930516a9930170afd93b71f860304e8c773d75bc3298684ac930dc"}
TMUX_STARTED {
  "tmux_socket": "marso-validation-20260915",
  "tmux_session": "stress_hard_20260915_045607",
  "launch_root": "/content/marso-tmux-launch/stress_hard_20260915_045607",
  "stdout": "/content/marso-tmux-launch/stress_hard_20260915_045607/tmux.log",
  "exit_file": "/content/marso-tmux-launch/stress_hard_20260915_045607/exit_code.txt",
  "manifest": "/content/marso-recovered-inputs/20260915-045535/validation-input.json",
  "wrapper_sha256": "0f1ee62503930516a9930170afd93b71f860304e8c773d75bc3298684ac930dc",
  "stages": "stress/hard",
  "command": [
    "/content/marso-py312/bin/python",
    "-u",
    "/content/berlin-marso-hackathon/tools/run_validation_durable.py",
    "--manifest",
    "/content/marso-recovered-inputs/20260915-045535/validation-input.json",
    "--out",
    "/content/marso-durable-v

In [4]:
import json, subprocess
from pathlib import Path
print('TMUX_ALIVE', subprocess.run(['tmux','-L',TMUX_SOCKET,'has-session','-t',TMUX_SESSION],capture_output=True).returncode==0)
lines = (LAUNCH_ROOT/'tmux.log').read_text().splitlines()
launch_header = next(json.loads(line) for line in lines if line.startswith('{') and 'local_dir' in json.loads(line))
DURABLE_ROOT = Path(launch_header['local_dir']); DURABLE_REMOTE = Path(launch_header['remote_dir'])
status = json.loads((DURABLE_ROOT/'records/status.json').read_text())
print('DURABLE_ROOT', DURABLE_ROOT); print('DURABLE_REMOTE', DURABLE_REMOTE)
print(json.dumps({k: status.get(k) for k in ['status','compute_status','remote_status','phase','elapsed_seconds','stages','copy_errors']}, indent=2))
print('CHILD_LOG_TAIL', (DURABLE_ROOT/'records/child.log').read_text()[-2000:])
p = LAUNCH_ROOT/'exit_code.txt'; print('EXIT_CODE', p.read_text().strip() if p.exists() else 'running')
print(subprocess.run(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used','--format=csv,noheader'],text=True,capture_output=True,timeout=10).stdout)


TMUX_ALIVE True


StopIteration: 